# PydanticAI 입문 가이드

이 노트북은 `analysis_agent.ipynb`, `automation_agent.ipynb`, `planner_agent.ipynb`에서 사용하는 **PydanticAI**의 핵심 개념을 단계별로 설명합니다.

## PydanticAI란?

PydanticAI는 LLM 기반 애플리케이션을 만들기 위한 Python 프레임워크입니다.

핵심 아이디어는 간단합니다:
1. **Agent** — LLM을 감싸는 객체. "누구에게 어떤 일을 시킬지" 정의
2. **Instructions** — Agent에게 주는 시스템 프롬프트 (역할, 규칙)
3. **Structured Output** — LLM 응답을 Pydantic 모델로 강제하여 JSON 형태로 받기
4. **Tools** — Agent가 호출할 수 있는 Python 함수 (검색, API 호출 등)

아래에서 하나씩 실습해보겠습니다.

## 1단계: 환경 설정

먼저 API 키를 로드하고 필요한 라이브러리를 임포트합니다.

- `dotenv`: `.env` 파일에 저장된 `OPENAI_API_KEY`를 환경변수로 로드
- `pydantic`: 구조화된 출력 스키마를 정의하는 데이터 검증 라이브러리
- `pydantic_ai`: LLM Agent를 만들고 실행하는 프레임워크

In [2]:
# .env 파일에서 OPENAI_API_KEY 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel   # 구조화된 출력 스키마 정의용
from pydantic_ai import Agent    # LLM 기반 Agent 프레임워크

## 2단계: 가장 간단한 Agent

PydanticAI에서 Agent를 만드는 최소 코드는 단 2줄입니다.

```python
agent = Agent("openai:gpt-5.4")          # 1) Agent 생성 (사용할 LLM 지정)
result = await agent.run("안녕하세요!")    # 2) 실행
```

- `Agent(모델명)`: 어떤 LLM을 사용할지 지정합니다
- `await agent.run(프롬프트)`: Agent에게 메시지를 보내고 응답을 받습니다
- `result.output`: 응답 텍스트가 담겨 있습니다

> **주의:** Jupyter 노트북에서는 `await agent.run()`을 사용합니다.
> 일반 Python 스크립트에서는 `agent.run_sync()`를 사용하면 됩니다.

---

# 실습1

In [3]:
agent = Agent("openai:gpt-5.4")    # 1) Agent 생성 *사용할 LLM 모델 지정
result = await agent.run("포스코DX는 어떤 회사야?")  # 2) Agent 실행 *프롬프트 입력
print(result)

AgentRunResult(output='포스코DX는 **포스코그룹의 IT·자동화·엔지니어링 전문 회사**입니다.\n\n간단히 말하면,  \n**공장·물류·에너지 설비를 더 똑똑하게 운영하도록 돕는 디지털 전환(DX, Digital Transformation) 회사**예요.\n\n## 핵심적으로 하는 일\n포스코DX는 주로 이런 사업을 합니다.\n\n- **스마트팩토리**\n  - 철강, 2차전지, 제조 공장의 자동화\n  - 생산관리시스템, 설비 제어, 데이터 분석\n\n- **산업 자동화**\n  - 공장 설비 제어 시스템\n  - 전기·계장·제어 엔지니어링\n  - PLC, SCADA 같은 산업제어 분야\n\n- **IT 서비스**\n  - 기업용 시스템 구축·운영\n  - 클라우드, AI, 빅데이터, 보안\n  - ERP, MES 등 기업/생산 시스템\n\n- **스마트 물류·스마트 인프라**\n  - 물류 자동화\n  - 항만, 철도, 교통 등 인프라 디지털화\n  - CCTV, 관제, 운영 시스템\n\n## 어떤 회사에서 바뀐 거야?\n포스코DX는 원래 **포스코ICT**라는 이름이었고,  \n**2023년에 ‘포스코DX’로 사명을 변경**했습니다.\n\n이름을 바꾼 이유는 단순 IT회사를 넘어서  \n**디지털 전환 중심 회사**라는 이미지를 더 강하게 가져가려는 의미가 큽니다.\n\n## 주요 특징\n- **포스코그룹 계열사**\n- 철강 분야에서 쌓은 **산업 현장 자동화 경험**이 강점\n- 최근에는 철강뿐 아니라 **배터리, 물류, 스마트시티, AI** 쪽으로도 확대 중\n\n## 한 줄로 정리\n**포스코DX는 포스코그룹의 스마트팩토리·산업자동화·IT서비스를 맡는 디지털 전환 전문 기업**입니다.\n\n원하시면 제가 이어서  \n**“포스코DX의 사업분야를 취업 준비 관점에서 설명”**하거나  \n**“포스코DX와 포스코홀딩스/포스코인터내셔널 차이”**도 정리해드릴게요.')


---

In [4]:
# 가장 간단한 Agent: 모델만 지정하고 바로 실행
simple_agent = Agent("openai:gpt-5.4")

# Agent에게 메시지를 보내고 응답 받기
result = await simple_agent.run("PydanticAI를 한 줄로 설명해줘.")

# result.output에 LLM의 응답 텍스트가 담겨 있음
print(result.output)

PydanticAI는 **Pydantic의 타입 검증 강점을 활용해 LLM 애플리케이션을 더 안전하고 구조적으로 개발할 수 있게 해주는 Python 에이전트 프레임워크**입니다.


## 3단계: Instructions (시스템 프롬프트)

`instructions` 파라미터로 Agent에게 **역할과 규칙**을 부여할 수 있습니다.
이는 ChatGPT의 "system prompt"와 같은 역할입니다.

```python
agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 해줘."
)
```

`instructions`가 있으면 Agent는 매 요청마다 이 규칙을 따릅니다.

In [5]:
# instructions로 Agent에게 역할 부여
chef_agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 친절한 요리사야. 모든 답변을 요리 비유로 설명해줘. 한국어로 답변.",
)

# 같은 질문이라도 instructions에 따라 답변 스타일이 달라짐
result = await chef_agent.run("소프트웨어 테스트가 왜 중요한가요?")
print(result.output)

소프트웨어 테스트는 요리로 치면 **손님에게 음식을 내기 전에 맛보고, 익힘 상태를 확인하고, 위생까지 점검하는 과정**이에요.

왜 중요하냐면요:

1. **버그를 미리 잡을 수 있어요**  
   국에 소금을 너무 많이 넣었는데 손님상에 나가기 전에 알아채면 바로 고칠 수 있죠.  
   테스트도 마찬가지로, 프로그램의 문제를 미리 발견해서 큰 사고를 막아줘요.

2. **품질을 일정하게 유지해요**  
   맛집은 매번 같은 맛을 내야 하잖아요.  
   소프트웨어도 업데이트할 때마다 기능이 제대로 동작하는지 확인해야 사용자들이 믿고 쓸 수 있어요.

3. **수리 비용을 줄여줘요**  
   요리를 다 만든 뒤에 재료가 상한 걸 발견하면 처음부터 다시 해야 할 수도 있어요.  
   소프트웨어도 출시 후 문제를 고치려면 시간과 돈이 훨씬 더 많이 들어요.

4. **안전과 신뢰를 지켜줘요**  
   음식에 알레르기 유발 재료가 잘못 들어가면 큰일 나죠.  
   소프트웨어도 금융, 의료, 자동차 같은 분야에서는 작은 오류가 큰 피해로 이어질 수 있어요.

5. **변경해도 안심할 수 있어요**  
   새로운 메뉴를 추가할 때 기존 인기 메뉴 맛이 변하면 안 되잖아요.  
   테스트는 새 기능을 넣어도 기존 기능이 망가지지 않았는지 확인해줘요.

한마디로 말하면,  
**소프트웨어 테스트는 “요리를 손님에게 내기 전 마지막으로 맛보고 점검하는 일”**이에요.  
이 과정이 없으면 보기엔 멀쩡해도 먹을 수 없는 음식이 나갈 수 있듯, 테스트 없는 소프트웨어도 겉만 멀쩡하고 실제로는 문제가 많을 수 있어요.

원하시면 제가 이것도  
- **개발자 관점**  
- **회사 비용 관점**  
- **초보자도 이해하기 쉬운 예시 중심**  
으로 더 풀어드릴게요.


---

## 실습

In [6]:
# instructions로 Agent에게 역할 부여
yeosu_tourist_agent = Agent(
    "openai:gpt-5.4",
    instructions="너는 대한민국 여수시의 여행 전문가야. 한국어로 답변.",
)

# 같은 질문이라도 instructions에 따라 답변 스타일이 달라짐
yeosu_tourist_result = await yeosu_tourist_agent.run("여수시에 가볼 만한 곳이 어디야? " \
"단 주어진 시간은 오후 1시부터 7시정도야. 관광지, 맛집, 카페 등 다양하게 알려줘.")
print(yeosu_tourist_result.output)

오후 1시부터 7시까지면 **여수 핵심 코스**를 꽤 알차게 즐길 수 있어요.  
시간이 아주 길진 않아서 **이동이 편한 곳 위주로 묶어서** 가는 게 좋아요.  
여수는 크게 **오동도·엑스포·해양공원 쪽**, **이순신광장·중앙동 쪽**, **돌산 쪽**으로 나눠 생각하면 편합니다.

아래에 **관광지 + 맛집 + 카페**를 함께 섞어서 추천드릴게요.

---

# 1. 오후 1시~7시 추천 코스 1: 여수 첫 방문자용 대표 코스
**이순신광장 → 중앙동/고소동 벽화마을 → 카페 → 돌산공원/해상케이블카 → 저녁식사**

## 1) 이순신광장
여수 오면 가장 무난하게 들르기 좋은 곳이에요.  
바다도 가깝고, 주변에 먹거리와 산책 포인트가 많아서 **짧은 시간 여행 시작점**으로 좋습니다.

- 볼거리: 이순신 장군 동상, 바다 풍경, 주변 골목
- 장점: 주변 맛집, 카페 밀집
- 추천 시간: 30분~1시간

### 근처 먹거리
- **여수당**: 쑥 아이스크림, 디저트류로 유명
- **바게트버거/핫도그류 간식집들**: 간단히 먹기 좋음
- **해물삼합, 게장, 서대회무침, 갈치조림 식당들** 다수

---

## 2) 고소동 벽화마을
이순신광장에서 멀지 않아서 같이 묶기 좋아요.  
벽화와 골목, 바다 내려다보는 풍경이 꽤 예쁩니다.

- 사진 찍기 좋음
- 언덕이 있어서 너무 덥거나 비 오면 조금 힘들 수 있음
- 추천 시간: 40분~1시간

---

## 3) 오션뷰 카페
여수는 바다 보이는 카페가 강점이에요.  
오후 시간대에 잠깐 쉬면서 여유 즐기기 좋습니다.

### 추천 스타일
- **돌산대교뷰 카페**
- **바다 바로 앞 대형 카페**
- **루프탑 카페**

### 카페 고를 때 팁
- 이동 동선상 **돌산 넘어가기 전/후**로 잡으면 좋음
- 주말에는 대형 카페가 주차가 더 편한 편
- 1시간 정도 잡으면 충분

---

## 4) 돌산공원 + 여수해상케이블카
짧은 시간 여수에서 “여행 온 느낌”을 가장 크게 주는 곳 중 하나예요.


---

## 4단계: Structured Output (구조화된 출력)

기본 Agent는 자유 텍스트를 반환합니다. 하지만 실무에서는 **정해진 형식의 데이터**가 필요합니다.

`output_type`에 Pydantic 모델을 지정하면, LLM이 자유 텍스트 대신 **해당 스키마에 맞는 데이터**를 반환합니다.

### 왜 필요할까요?

| 자유 텍스트 | 구조화된 출력 |
|---|---|
| "이 영화는 8점입니다" | `{"title": "...", "score": 8, "genre": "SF"}` |
| 후속 처리 어려움 | DB 저장, API 연동 바로 가능 |

### 사용법

```python
class Movie(BaseModel):        # 1) 원하는 출력 형식을 Pydantic 모델로 정의
    title: str
    score: int

agent = Agent(
    "openai:gpt-5.4",
    output_type=Movie,          # 2) Agent에 output_type으로 지정
)
result = await agent.run(...)
movie = result.output           # 3) result.output이 Movie 타입으로 반환됨
print(movie.title)              #    → 속성으로 바로 접근 가능
```

> **실전 활용:** `analysis_agent.ipynb`에서 `CodeReview` 모델로 코드 리뷰 결과를, `automation_agent.ipynb`에서 `DailyReport` 모델로 일일 보고서를 구조화합니다.

In [7]:
# Step 1) 원하는 출력 형식을 Pydantic 모델로 정의
class MovieReview(BaseModel):
    title: str       # 영화 제목
    score: int       # 평점 (1~10)
    pros: list[str]  # 장점 목록
    cons: list[str]  # 단점 목록


# Step 2) output_type으로 지정 → LLM이 반드시 이 형식으로 응답
review_agent = Agent(
    "openai:gpt-5.4",
    output_type=MovieReview,
    instructions="영화 평론가로서 요청받은 영화를 평가. 한국어로 작성.",
)

# Step 3) 실행 → result.output이 MovieReview 타입
result = await review_agent.run("인터스텔라를 평가해주세요.")
review = result.output

# Pydantic 모델이므로 속성으로 바로 접근 가능
print(f"제목: {review.title}")
print(f"평점: {review.score}/10")
print(f"장점: {', '.join(review.pros)}")
print(f"단점: {', '.join(review.cons)}")

제목: 인터스텔라
평점: 93/10
장점: 웅장한 우주 스케일과 인간적인 가족 서사를 균형 있게 결합한다., 크리스토퍼 놀란 특유의 구조적 연출과 시간 개념의 활용이 강한 몰입감을 만든다., 한스 치머의 음악과 실용효과 중심의 비주얼이 압도적인 체험을 제공한다., 매튜 맥커너헤이와 앤 해서웨이의 감정 연기가 이야기의 중심을 단단히 붙든다.
단점: 중반 이후 과학적 개념 설명이 다소 직설적이라 관객에 따라 과잉 친절하게 느껴질 수 있다., 철학적·감성적 결론이 하드 SF를 기대한 관객에게는 다소 비약적으로 보일 수 있다., 러닝타임이 길고 전개가 묵직해 가벼운 감상을 원하는 관객에게는 부담이 있다.


## 5단계: Tools (도구)

Agent에게 **Python 함수를 도구로 등록**하면, LLM이 필요할 때 해당 함수를 호출할 수 있습니다.

### 왜 필요할까요?

LLM은 혼자서는 다음을 할 수 없습니다:
- 실시간 데이터 조회 (날씨, 주가, DB)
- 외부 시스템 조작 (이메일 발송, 파일 저장)
- 계산이나 코드 실행

Tool을 등록하면 LLM이 **"이 함수를 호출해야겠다"고 판단**하고, 프레임워크가 실제 함수를 실행합니다.

### 사용법

```python
@agent.tool_plain          # 데코레이터로 도구 등록
def add(a: int, b: int) -> str:
    """두 숫자를 더한다."""     # docstring이 LLM에게 도구 설명으로 전달됨
    return str(a + b)
```

- `@agent.tool_plain`: Agent 컨텍스트 없이 독립적으로 동작하는 도구
- **함수 이름과 docstring**이 LLM에게 전달되어, LLM이 언제 이 도구를 쓸지 판단
- **타입 힌트** (`a: int, b: int`)가 LLM에게 파라미터 형식을 알려줌

> **실전 활용:** `automation_agent.ipynb`에서 `fetch_github_issues`, `send_slack_message`를, `planner_agent.ipynb`에서 `search_web`, `write_section`을 도구로 등록합니다.

In [8]:
# Tool이 있는 Agent 만들기: 간단한 계산기 예제
calc_agent = Agent(
    "openai:gpt-5.4",
    instructions="계산이 필요하면 반드시 제공된 도구를 사용하라. 한국어로 답변.",
)


# @agent.tool_plain 데코레이터로 도구 등록
# - 함수 이름(add)과 docstring이 LLM에게 전달됨
# - LLM이 "덧셈이 필요하다"고 판단하면 이 함수를 호출
@calc_agent.tool_plain
def add(a: int, b: int) -> int:
    """두 숫자를 더한다."""
    print(f"  [Tool 호출] add({a}, {b})")
    return a + b


@calc_agent.tool_plain
def multiply(a: int, b: int) -> int:
    """두 숫자를 곱한다."""
    print(f"  [Tool 호출] multiply({a}, {b})")
    return a * b


# Agent 실행 — LLM이 알아서 필요한 도구를 선택하여 호출
result = await calc_agent.run("17과 28을 더하고, 그 결과에 3을 곱해주세요.")
print(f"\n최종 답변: {result.output}")

  [Tool 호출] add(17, 28)
  [Tool 호출] multiply(45, 3)

최종 답변: 결과는 **135**입니다.


## 6단계: 모두 합치기 — Tool + Structured Output

지금까지 배운 것을 조합하면 **실전에서 쓸 수 있는 Agent**가 됩니다.

이 예제에서는 도시 이름을 받아 날씨를 조회(Tool)하고, 구조화된 여행 추천(Structured Output)을 반환하는 Agent를 만듭니다.

```
사용자: "서울 여행 추천해줘"
   ↓
Agent가 get_weather("서울") Tool 호출
   ↓
LLM이 날씨 정보를 참고해 TravelAdvice 형식으로 응답
   ↓
result.output → TravelAdvice(city="서울", weather="맑음", ...)
```

In [9]:
# --- 구조화된 출력 정의 ---
class TravelAdvice(BaseModel):
    city: str              # 도시 이름
    weather: str           # 현재 날씨
    recommended: bool      # 여행 추천 여부
    tips: list[str]        # 여행 팁 목록


# --- Agent 생성: Tool + Structured Output 조합 ---
travel_agent = Agent(
    "openai:gpt-5.4",
    output_type=TravelAdvice,  # 구조화된 형식으로 응답 강제
    instructions=(
        "여행 가이드로서 도시의 날씨를 조회하고 여행 조언을 제공한다. "
        "반드시 get_weather 도구로 날씨를 확인한 후 답변하라. 한국어로 작성."
    ),
)


# --- Tool 등록: 날씨 조회 (Mock) ---
@travel_agent.tool_plain
def get_weather(city: str) -> str:
    """도시의 현재 날씨 정보를 조회한다."""
    # 실제로는 날씨 API를 호출하지만, 여기서는 Mock 데이터 반환
    mock_data = {
        "서울": "맑음, 22°C, 습도 45%",
        "부산": "흐림, 19°C, 오후 비 예보",
        "제주": "맑음, 24°C, 바람 강함",
    }
    weather = mock_data.get(city, f"{city}: 데이터 없음, 약 20°C 예상")
    print(f"  [Tool 호출] get_weather('{city}') → {weather}")
    return weather


# --- 실행 ---
result = await travel_agent.run("제주도 여행을 계획 중이에요. 지금 가도 될까요?")
advice = result.output

print(f"도시: {advice.city}")
print(f"날씨: {advice.weather}")
print(f"추천: {'✅ 추천' if advice.recommended else '❌ 비추천'}")
print("팁:")
for tip in advice.tips:
    print(f"  - {tip}")

  [Tool 호출] get_weather('제주도') → 제주도: 데이터 없음, 약 20°C 예상
도시: 제주도
날씨: 데이터 없음, 약 20°C 예상
추천: ✅ 추천
팁:
  - 현재 상세 실시간 데이터는 없지만 약 20°C로 예상되어 여행하기에는 무난한 편입니다.
  - 제주도는 바람이 강하고 날씨 변화가 잦으니 가벼운 겉옷을 꼭 챙기세요.
  - 비 가능성에 대비해 우산이나 방수 재킷을 준비하면 좋습니다.
  - 야외 일정과 실내 일정을 함께 짜 두면 변덕스러운 날씨에 대응하기 좋습니다.


## 정리: PydanticAI 핵심 패턴 요약

| 개념 | 역할 | 코드 |
|------|------|------|
| **Agent** | LLM을 감싸는 핵심 객체 | `Agent("openai:gpt-5.4")` |
| **instructions** | 역할·규칙 부여 (시스템 프롬프트) | `Agent(..., instructions="...")` |
| **output_type** | 응답을 Pydantic 모델로 강제 | `Agent(..., output_type=MyModel)` |
| **@agent.tool_plain** | LLM이 호출할 수 있는 함수 등록 | `@agent.tool_plain` 데코레이터 |
| **await agent.run()** | Agent 실행 (Jupyter용) | `result = await agent.run("...")` |
| **result.output** | 응답 데이터 접근 | `result.output.title` |

### 다음 단계

이 개념들이 실전에서 어떻게 조합되는지 확인해보세요:

| 노트북 | Agent 유형 | 핵심 패턴 |
|--------|-----------|----------|
| `analysis_agent.ipynb` | 분석형 | output_type만 사용 (Tool 없음) |
| `automation_agent.ipynb` | 자동화형 | Tool + output_type (고정 순서) |
| `planner_agent.ipynb` | Planner형 | Tool만 사용 (LLM이 순서 결정) |